# Gradient descent, learning rate and convexity

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea
You are standing on a foggy mountain and want to reach the valley. All you can feel is the slope under your feet. The best move: take **one step downhill** and repeat.

$$\theta\leftarrow\theta-\eta\,\nabla L(\theta)$$

| Symbol | Name | Role |
|---|---|---|
| $\theta$ | Parameters | What is being learned |
| $\nabla L$ | Gradient | The uphill direction |
| $\eta$ | Learning rate | The step size |

## A complete numeric example
Minimize $f(x)=(x-3)^2$ from $x_0=0$ with $\eta=0.1$. The derivative is $f'(x)=2(x-3)$.

| Step | $x$ | $f'(x)$ | $x-\eta f'(x)$ | new $f$ |
|---|---|---|---|---|
| 1 | 0 | −6 | 0.6 | 5.76 |
| 2 | 0.6 | −4.8 | 1.08 | 3.686 |
| 3 | 1.08 | −3.84 | 1.464 | 2.359 |
| 4 | 1.464 | −3.072 | 1.771 | 1.510 |
| 5 | 1.771 | −2.458 | 2.017 | 0.966 |

The distance to the answer ($x=3$) is multiplied by $1-2\eta=0.8$ at every step: $3\to2.4\to1.92\to\dots$, so convergence is **geometric**.

## The learning rate: the most important setting
> [Three learning rates on $f(x)=x^2$ from the same starting point.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_lr.png)

*Three learning rates on $f(x)=x^2$ from the same starting point.*

| $\eta$ | Behaviour | Sign in the loss curve |
|---|---|---|
| Too small | Very slow convergence | The loss creeps down |
| Good | Fast and stable | A smooth decrease |
| A bit large | Oscillates around the minimum | Goes up and down |
| Too large | Diverges | The loss grows or becomes nan |

For $f(x)=x^2$ each step is $x\leftarrow(1-2\eta)x$, so it converges only when $\lvert1-2\eta\rvert<1$, that is $0<\eta<1$. In general the limit is $2/\lambda_{\max}$, where $\lambda_{\max}$ is the largest eigenvalue of the Hessian.

## In two dimensions: narrow valleys
> Open the interactive lab for this part on the website: https://behrooztaheri.github.io/learn-python-ai/#/l/math-gd

Choose the "narrow valley" in the lab. Because the slope along y is ten times the slope along x, the path **zigzags** and moves slowly. Increase the momentum: the zigzags cancel. This happens when features have different scales, which is one more reason to **scale your features**.

## Convex and non-convex functions
> [Left: a convex function has a single minimum. Right: a non-convex function has local minima.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_convex.png)

*Left: a convex function has a single minimum. Right: a non-convex function has local minima.*

| | Convex | Non-convex |
|---|---|---|
| Shape | A bowl | A mountain range |
| Local minimum | Is the global one | May not be global |
| Guarantee for GD | Reaches the best answer | Depends on the starting point |
| Examples | Linear and logistic regression, SVM | Neural networks |

## Three versions of gradient descent
| Version | Data per step | Advantage | Drawback |
|---|---|---|---|
| Batch | All n samples | Exact gradient | Slow on large data |
| SGD (stochastic) | 1 sample | Fast, can escape local minima | Noisy |
| Mini-batch | 32 to 256 samples | A balance; suits GPUs | One more hyperparameter |

One **epoch** is one pass over all the data. With 1000 samples and batches of 100, each epoch has ten steps.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1) The table from the lesson: f(x) = (x - 3)^2
x, eta = 0.0, 0.1
for step in range(1, 6):
    grad = 2 * (x - 3)
    x = x - eta * grad
    print(f"step {step}: grad = {grad:7.3f}   x = {x:.3f}   f = {(x - 3) ** 2:.3f}")

# 2) Effect of the learning rate on f(x) = x^2
for eta in [0.05, 0.4, 0.95, 1.05]:
    x = 2.5
    for _ in range(20): x -= eta * 2 * x
    print(f"eta = {eta:<5} after 20 steps x = {x:.4g}")

# 3) Mini-batch gradient descent for linear regression  y = 4 + 3x + noise
rng = np.random.default_rng(0)
X = rng.uniform(0, 2, 200); y = 4 + 3 * X + rng.normal(0, 0.5, 200)
w, b, eta, history = 0.0, 0.0, 0.1, []
for epoch in range(30):
    order = rng.permutation(200)
    for start in range(0, 200, 20):                  # batches of 20
        i = order[start:start + 20]
        err = w * X[i] + b - y[i]
        w -= eta * 2 * np.mean(err * X[i])
        b -= eta * 2 * np.mean(err)
    history.append(np.mean((w * X + b - y) ** 2))
print(f"learned w = {w:.3f}, b = {b:.3f}   (true 3 and 4)")

plt.plot(history, marker="o"); plt.xlabel("epoch"); plt.ylabel("MSE"); plt.yscale("log")
plt.title("Training loss"); plt.grid(alpha=0.3); plt.show()

### Exercise

For $f(x)=x^2$ with $x_0=4$ and $\eta=0.25$, do three steps by hand. What do you notice?

In [ ]:
# Your answer here

### Exercise

Choose "two minima" in the lab and run from two different starting points. Do you reach the same answer?

In [ ]:
# Your answer here